# Week 2 Day 3: LangGraph Stateful Workflows

## Task 1: Graph Concepts and State Design

LangGraph models an agent workflow as a **stateful graph**. Each step is explicit, so the workflow can branch, loop, pause, and resume instead of hiding control flow inside one large agent loop.

### Core building blocks

- **`StateGraph`**: The graph builder. It receives the schema for the shared state and is used to register nodes and connect transitions before compilation.
- **Nodes**: Python functions that perform one unit of work. A node reads the current state and returns either a partial state update or the next state values.
- **Edges**: Fixed transitions between nodes. An edge says which node should run after the current node completes.
- **Conditional edges**: Routing functions that inspect the current state and choose the next node. They are useful for decisions such as `needs_revision` versus `approved`.
- **Shared `State` object**: The single, typed context passed through the workflow. It stores the user request, research findings, draft, critique, revision count, and completion status. Nodes communicate through this object instead of relying on hidden global variables.

### Workflow choice: research assistant

The assistant will search for evidence, draft an answer, critique that draft, and revise it when the critique identifies problems. The revision loop is bounded by `max_revisions` so that a bad or incomplete critique cannot create an infinite graph execution.

### Graph design before coding

```text
                    +----------------+
                    |  START / query |
                    +-------+--------+
                            |
                            v
                    +----------------+
                    |     search     |
                    +-------+--------+
                            |
                            v
                    +----------------+
                    |      draft     |
                    +-------+--------+
                            |
                            v
                    +----------------+
                    |    critique    |
                    +-------+--------+
                            |
                 +----------+----------+
                 |                     |
       needs revision?             approved
                 |                     |
                 v                     v
          +-------------+      +-------------+
          |   revise    |      |     END     |
          +------+------+      +-------------+
                 |
                 +------> critique
```

The `revise -> critique` edge creates the cycle. The conditional router will send the workflow to `revise` only while the answer needs work and the revision budget remains; otherwise it will terminate.


In [1]:
from typing import Literal, TypedDict


class ResearchState(TypedDict):
    """Shared state passed between the research workflow nodes."""

    question: str
    search_queries: list[str]
    sources: list[dict[str, str]]
    draft: str
    critique: str
    revision_count: int
    max_revisions: int
    status: Literal["searching", "drafting", "critiquing", "revising", "approved"]
    needs_revision: bool


initial_state: ResearchState = {
    "question": "How does LangGraph support cyclical agent workflows?",
    "search_queries": [],
    "sources": [],
    "draft": "",
    "critique": "",
    "revision_count": 0,
    "max_revisions": 2,
    "status": "searching",
    "needs_revision": False,
}

initial_state

{'question': 'How does LangGraph support cyclical agent workflows?',
 'search_queries': [],
 'sources': [],
 'draft': '',
 'critique': '',
 'revision_count': 0,
 'max_revisions': 2,
 'status': 'searching',
 'needs_revision': False}

## Task 2: Build a Linear Graph

This graph follows a fixed sequence:

```text
START -> plan -> retrieve -> generate -> format -> END
```

Each node returns only the fields it updates. LangGraph merges those partial updates into the shared state before passing it to the next node.


In [2]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class LinearState(TypedDict, total=False):
    question: str
    plan: list[str]
    retrieved_context: list[str]
    draft: str
    answer: str


def plan_node(state: LinearState) -> dict[str, list[str]]:
    plan = [
        "Identify the main LangGraph concept",
        "Connect it to stateful workflows",
        "Present the result clearly",
    ]
    updated_state = {"plan": plan}
    print("After plan:", {**state, **updated_state})
    return updated_state


def retrieve_node(state: LinearState) -> dict[str, list[str]]:
    context = [
        "LangGraph represents workflows as nodes connected by edges.",
        "Nodes read shared state and return partial updates.",
    ]
    updated_state = {"retrieved_context": context}
    print("After retrieve:", {**state, **updated_state})
    return updated_state


def generate_node(state: LinearState) -> dict[str, str]:
    draft = (
        f"{state['question']} "
        f"It uses a graph of steps that share state. "
        f"Key evidence: {state['retrieved_context'][0]}"
    )
    updated_state = {"draft": draft}
    print("After generate:", {**state, **updated_state})
    return updated_state


def format_node(state: LinearState) -> dict[str, str]:
    answer = f"Answer: {state['draft']}"
    updated_state = {"answer": answer}
    print("After format:", {**state, **updated_state})
    return updated_state


workflow = StateGraph(LinearState)
workflow.add_node("plan", plan_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)
workflow.add_node("format", format_node)
workflow.add_edge(START, "plan")
workflow.add_edge("plan", "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "format")
workflow.add_edge("format", END)
linear_graph = workflow.compile()

sample_input: LinearState = {
    "question": "What makes LangGraph useful for agent workflows?"
}
final_state = linear_graph.invoke(sample_input)

print("\nFinal state:", final_state)

After plan: {'question': 'What makes LangGraph useful for agent workflows?', 'plan': ['Identify the main LangGraph concept', 'Connect it to stateful workflows', 'Present the result clearly']}
After retrieve: {'question': 'What makes LangGraph useful for agent workflows?', 'plan': ['Identify the main LangGraph concept', 'Connect it to stateful workflows', 'Present the result clearly'], 'retrieved_context': ['LangGraph represents workflows as nodes connected by edges.', 'Nodes read shared state and return partial updates.']}
After generate: {'question': 'What makes LangGraph useful for agent workflows?', 'plan': ['Identify the main LangGraph concept', 'Connect it to stateful workflows', 'Present the result clearly'], 'retrieved_context': ['LangGraph represents workflows as nodes connected by edges.', 'Nodes read shared state and return partial updates.'], 'draft': 'What makes LangGraph useful for agent workflows? It uses a graph of steps that share state. Key evidence: LangGraph represen

## Task 3: Conditional Edges and Cycles

The critique node uses a conditional edge to route back to `generate` when the quality score is below `0.8`, or to `finish` when the answer is acceptable. A retry counter and maximum retry limit make the cycle bounded and observable.

A plain `AgentExecutor` generally hides control flow inside an agent loop, so expressing and inspecting a specific critique-to-generation branch requires custom callbacks and stopping logic. LangGraph makes the branch explicit as a conditional edge, while the state object naturally carries the score, retry count, and loop history between passes.


In [3]:
from typing import TypedDict


class CritiqueState(TypedDict, total=False):
    question: str
    draft: str
    quality_score: float
    critique: str
    retry_count: int
    max_retries: int
    loop_log: list[str]
    answer: str
    status: str


def generate_with_retry(state: CritiqueState) -> dict[str, object]:
    retry_count = state.get("retry_count", 0)
    loop_log = state.get("loop_log", [])

    if retry_count == 0:
        draft = "LangGraph uses graphs."
    else:
        draft = (
            "LangGraph models agent workflows as explicit graphs of nodes and edges. "
            "Nodes read shared state, return updates, and can route conditionally "
            "to support controlled self-correction."
        )

    pass_number = retry_count + 1
    log_entry = f"Generate pass {pass_number} (retry_count={retry_count})"
    updated_state = {
        "draft": draft,
        "loop_log": [*loop_log, log_entry],
        "status": "generated",
    }
    print(log_entry, "->", draft)
    return updated_state


def critique_node(state: CritiqueState) -> dict[str, object]:
    retry_count = state.get("retry_count", 0)
    max_retries = state["max_retries"]
    loop_log = state.get("loop_log", [])

    if len(state["draft"]) < 60:
        quality_score = 0.4
        critique = "Add detail about nodes, edges, and shared state."
    else:
        quality_score = 0.9
        critique = "The answer explains the graph structure and state updates."

    log_entry = (
        f"Critique pass {retry_count + 1}: score={quality_score}, "
        f"retries={retry_count}/{max_retries}"
    )
    updated_state = {
        "quality_score": quality_score,
        "critique": critique,
        "loop_log": [*loop_log, log_entry],
        "status": "critiqued",
    }
    print(log_entry, "->", critique)
    return updated_state


def route_after_critique(state: CritiqueState) -> str:
    quality_is_acceptable = state["quality_score"] >= 0.8
    retries_are_exhausted = state.get("retry_count", 0) >= state["max_retries"]

    if quality_is_acceptable or retries_are_exhausted:
        return "finish"
    return "retry"


def record_retry(state: CritiqueState) -> dict[str, object]:
    retry_count = state.get("retry_count", 0) + 1
    print(f"Looping back to generate (retry {retry_count}/{state['max_retries']})")
    return {"retry_count": retry_count, "status": "retrying"}


def finish_node(state: CritiqueState) -> dict[str, str]:
    status = "approved" if state["quality_score"] >= 0.8 else "max retries reached"
    answer = f"{state['draft']}\n\nCritique: {state['critique']}"
    print(f"Finish: {status}")
    return {"answer": answer, "status": status}


cyclic_workflow = StateGraph(CritiqueState)
cyclic_workflow.add_node("generate", generate_with_retry)
cyclic_workflow.add_node("critique", critique_node)
cyclic_workflow.add_node("record_retry", record_retry)
cyclic_workflow.add_node("finish", finish_node)
cyclic_workflow.add_edge(START, "generate")
cyclic_workflow.add_edge("generate", "critique")
cyclic_workflow.add_conditional_edges(
    "critique",
    route_after_critique,
    {"retry": "record_retry", "finish": "finish"},
)
cyclic_workflow.add_edge("record_retry", "generate")
cyclic_workflow.add_edge("finish", END)
cyclic_graph = cyclic_workflow.compile()

cyclic_input: CritiqueState = {
    "question": "Why is LangGraph useful for agent workflows?",
    "max_retries": 2,
    "retry_count": 0,
    "loop_log": [],
}
cyclic_final_state = cyclic_graph.invoke(cyclic_input)

print("\nLoop log:")
for entry in cyclic_final_state["loop_log"]:
    print("-", entry)
print("\nFinal cyclic state:", cyclic_final_state)

Generate pass 1 (retry_count=0) -> LangGraph uses graphs.
Critique pass 1: score=0.4, retries=0/2 -> Add detail about nodes, edges, and shared state.
Looping back to generate (retry 1/2)
Generate pass 2 (retry_count=1) -> LangGraph models agent workflows as explicit graphs of nodes and edges. Nodes read shared state, return updates, and can route conditionally to support controlled self-correction.
Critique pass 2: score=0.9, retries=1/2 -> The answer explains the graph structure and state updates.
Finish: approved

Loop log:
- Generate pass 1 (retry_count=0)
- Critique pass 1: score=0.4, retries=0/2
- Generate pass 2 (retry_count=1)
- Critique pass 2: score=0.9, retries=1/2

Final cyclic state: {'question': 'Why is LangGraph useful for agent workflows?', 'draft': 'LangGraph models agent workflows as explicit graphs of nodes and edges. Nodes read shared state, return updates, and can route conditionally to support controlled self-correction.', 'quality_score': 0.9, 'critique': 'The ans

## Task 4: Human-in-the-Loop and Interrupts

This workflow prepares a purchase request and pauses immediately before the risky `make_purchase` node. A human reviewer updates the checkpoint with either approval or rejection, and the graph then resumes from the paused point.

Human approval should be required when an action can cause financial loss, legal or compliance exposure, privacy harm, irreversible changes, or meaningful impact on a person. Full autonomy is more appropriate for low-risk, reversible, well-monitored tasks with clear limits, such as drafting text, classifying routine data, or preparing a recommendation that a user can review later.


In [4]:
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver


class PurchaseState(TypedDict, total=False):
    item: str
    amount: float
    approved: bool
    review_note: str
    result: str
    status: str


def prepare_purchase(state: PurchaseState) -> dict[str, str]:
    request = f"Purchase {state['item']} for ${state['amount']:.2f}"
    print("Prepared:", request)
    return {"status": "awaiting_human_approval"}


def make_purchase(state: PurchaseState) -> dict[str, str]:
    if state.get("approved", False):
        result = f"SIMULATED PURCHASE COMPLETED: {state['item']}"
        status = "purchase_completed"
    else:
        result = f"PURCHASE REJECTED: {state['review_note']}"
        status = "purchase_rejected"

    print(result)
    return {"result": result, "status": status}


approval_checkpointer = MemorySaver()
approval_workflow = StateGraph(PurchaseState)
approval_workflow.add_node("prepare_purchase", prepare_purchase)
approval_workflow.add_node("make_purchase", make_purchase)
approval_workflow.add_edge(START, "prepare_purchase")
approval_workflow.add_edge("prepare_purchase", "make_purchase")
approval_workflow.add_edge("make_purchase", END)
approval_graph = approval_workflow.compile(
    checkpointer=approval_checkpointer,
    interrupt_before=["make_purchase"],
)


def run_human_review(thread_id: str, approved: bool, review_note: str) -> dict[str, object]:
    config = {"configurable": {"thread_id": thread_id}}
    initial_request: PurchaseState = {
        "item": "team productivity software",
        "amount": 249.00,
    }

    paused_state = approval_graph.invoke(initial_request, config)
    checkpoint = approval_graph.get_state(config)
    print("Paused before risky action:", checkpoint.next)

    approval_graph.update_state(
        config,
        {"approved": approved, "review_note": review_note},
    )
    resumed_state = approval_graph.invoke(None, config)
    print("Resumed state:", resumed_state)
    return resumed_state


print("--- Simulated human approval ---")
approved_result = run_human_review(
    thread_id="purchase-approved",
    approved=True,
    review_note="Budget and vendor have been verified.",
)

print("\n--- Simulated human rejection ---")
rejected_result = run_human_review(
    thread_id="purchase-rejected",
    approved=False,
    review_note="The request exceeds the current budget.",
)

assert approved_result["status"] == "purchase_completed"
assert rejected_result["status"] == "purchase_rejected"

--- Simulated human approval ---
Prepared: Purchase team productivity software for $249.00
Paused before risky action: ('make_purchase',)
SIMULATED PURCHASE COMPLETED: team productivity software
Resumed state: {'item': 'team productivity software', 'amount': 249.0, 'approved': True, 'review_note': 'Budget and vendor have been verified.', 'result': 'SIMULATED PURCHASE COMPLETED: team productivity software', 'status': 'purchase_completed'}

--- Simulated human rejection ---
Prepared: Purchase team productivity software for $249.00
Paused before risky action: ('make_purchase',)
PURCHASE REJECTED: The request exceeds the current budget.
Resumed state: {'item': 'team productivity software', 'amount': 249.0, 'approved': False, 'review_note': 'The request exceeds the current budget.', 'result': 'PURCHASE REJECTED: The request exceeds the current budget.', 'status': 'purchase_rejected'}


## Task 5: Persistence and Debugging

`SqliteSaver` stores graph checkpoints in a local SQLite file under a `thread_id`, allowing a paused conversation to be resumed after the checkpointer is closed and reopened. The state history exposes the checkpoints created during a run, which can be inspected or replayed from an earlier point to reproduce a debugging scenario.

### LangChain `AgentExecutor` versus LangGraph

Reach for **`AgentExecutor`** when the workflow is mostly a straightforward tool-using loop and you want a quick, conventional agent with minimal orchestration code. Reach for **LangGraph** when you need durable state, explicit branching or cycles, human approval, resumability, retries, or detailed inspection of intermediate checkpoints; its graph structure makes those production controls first-class.


In [1]:
import sqlite3
import uuid
from typing import TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph


class ConversationState(TypedDict, total=False):
    user_input: str
    messages: list[str]
    turn: int
    status: str


def capture_message(state: ConversationState) -> dict[str, object]:
    messages = [*state.get("messages", []), f"User: {state['user_input']}"]
    turn = state.get("turn", 0) + 1
    print(f"Captured turn {turn}")
    return {"messages": messages, "turn": turn, "status": "paused_for_response"}


def respond(state: ConversationState) -> dict[str, object]:
    response = (
        "LangGraph persists state as checkpoints keyed by thread_id, "
        "so a conversation can be resumed after a pause."
    )
    messages = [*state["messages"], f"Assistant: {response}"]
    print("Responded to turn", state["turn"])
    return {"messages": messages, "status": "completed"}


def build_conversation_graph(checkpointer: SqliteSaver):
    conversation_workflow = StateGraph(ConversationState)
    conversation_workflow.add_node("capture", capture_message)
    conversation_workflow.add_node("respond", respond)
    conversation_workflow.add_edge(START, "capture")
    conversation_workflow.add_edge("capture", "respond")
    conversation_workflow.add_edge("respond", END)
    return conversation_workflow.compile(
        checkpointer=checkpointer,
        interrupt_before=["respond"],
    )


checkpoint_path = "week2_day3_task5_checkpoints.sqlite"
thread_id = f"demo-conversation-{uuid.uuid4().hex[:8]}"
conversation_config = {"configurable": {"thread_id": thread_id}}

# Session 1: persist a paused conversation to disk.
first_connection = sqlite3.connect(checkpoint_path, check_same_thread=False)
first_checkpointer = SqliteSaver(first_connection)
first_checkpointer.setup()
first_graph = build_conversation_graph(first_checkpointer)
first_graph.invoke(
    {"user_input": "What is a LangGraph checkpoint?"},
    conversation_config,
)
paused = first_graph.get_state(conversation_config)
print("Paused checkpoint:", paused.values)
print("Next node:", paused.next)
first_connection.close()

# Session 2: reopen the SQLite file and resume the same thread.
second_connection = sqlite3.connect(checkpoint_path, check_same_thread=False)
second_checkpointer = SqliteSaver(second_connection)
second_graph = build_conversation_graph(second_checkpointer)
resumed = second_graph.invoke(None, conversation_config)
print("Resumed conversation:", resumed)

# Start a second turn on the same persisted thread and pause it again.
second_graph.invoke(
    {"user_input": "How does persistence help debugging?"},
    conversation_config,
)
second_pause = second_graph.get_state(conversation_config)
print("Second-turn checkpoint:", second_pause.values)

# Inspect the built-in checkpoint history for this thread.
history = list(second_graph.get_state_history(conversation_config))
print("\nCheckpoint history:")
for index, snapshot in enumerate(history, start=1):
    print(index, "next=", snapshot.next, "values=", snapshot.values)

# Replay the second turn from its paused checkpoint without changing the main thread.
replay_checkpoint = next(
    snapshot
    for snapshot in history
    if snapshot.next == ("respond",)
    and snapshot.values.get("user_input") == "How does persistence help debugging?"
)
replayed = second_graph.invoke(None, replay_checkpoint.config)
print("\nReplayed from checkpoint:", replayed)
second_connection.close()

assert paused.next == ("respond",)
assert resumed["status"] == "completed"
assert replayed["status"] == "completed"

Captured turn 1
Paused checkpoint: {'user_input': 'What is a LangGraph checkpoint?', 'messages': ['User: What is a LangGraph checkpoint?'], 'turn': 1, 'status': 'paused_for_response'}
Next node: ('respond',)
Responded to turn 1
Resumed conversation: {'user_input': 'What is a LangGraph checkpoint?', 'messages': ['User: What is a LangGraph checkpoint?', 'Assistant: LangGraph persists state as checkpoints keyed by thread_id, so a conversation can be resumed after a pause.'], 'turn': 1, 'status': 'completed'}
Captured turn 2
Second-turn checkpoint: {'user_input': 'How does persistence help debugging?', 'messages': ['User: What is a LangGraph checkpoint?', 'Assistant: LangGraph persists state as checkpoints keyed by thread_id, so a conversation can be resumed after a pause.', 'User: How does persistence help debugging?'], 'turn': 2, 'status': 'paused_for_response'}

Checkpoint history:
1 next= ('respond',) values= {'user_input': 'How does persistence help debugging?', 'messages': ['User: Wh